# DefectosCafeVerde — paired-side complementarity audit

Validation-only audit of the completed native D0DIRECT seed-42 endpoint. It measures whether the two physical sides of one bean contain complementary correct decisions and whether a learned dual-view model has headroom beyond the paper's max-confidence rule. No training is performed and test is never extracted.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, time
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/defectoscafeverde-dual-view-fusion'
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'
EXPECTED_BYTES=686474752
EXPECTED_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
drive_roots=[p for p in [Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')] if p.exists()]
archives=sorted({p for root in drive_roots for p in root.rglob(ARCHIVE_NAME) if p.is_file()})
valid=[p for p in archives if p.stat().st_size==EXPECTED_BYTES and sha256(p)==EXPECTED_SHA]
ARCHIVE_ANCHOR=valid[0] if valid else None
if valid:
    ARCHIVE=valid[0]
else:
    complete=[]
    for scheme,count in (('part16',41),('chunk',11),('part',2)):
        by_parent={}
        for root in drive_roots:
            for path in root.rglob(f'{ARCHIVE_NAME}.{scheme}-*'):
                if path.is_file(): by_parent.setdefault(path.parent,{})[path.name]=path
        expected=[f'{ARCHIVE_NAME}.{scheme}-{index:03d}' for index in range(count)]
        for parent,rows in by_parent.items():
            if sorted(rows)==expected and sum(rows[name].stat().st_size for name in expected)==EXPECTED_BYTES: complete.append([rows[name] for name in expected])
    if not complete: raise FileNotFoundError(f'Tidak menemukan {ARCHIVE_NAME} atau set chunk lengkap di shared Drive')
    parts=complete[0]; ARCHIVE_ANCHOR=parts[0]; ARCHIVE=WORK/ARCHIVE_NAME
    with ARCHIVE.open('wb') as target:
        for part in parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if ARCHIVE.stat().st_size!=EXPECTED_BYTES or sha256(ARCHIVE)!=EXPECTED_SHA: raise RuntimeError('Archive gabungan tidak valid')
result_rel=Path('experiments/defectoscafeverde-cwcf-direct-v1/val_reports/D0DIRECT_seed42_result.json')
checkpoint_rel=Path('experiments/defectoscafeverde-cwcf-direct-v1/D0DIRECT/D0DIRECT_seed42/weights/best.pt')
options=[Path('/content/drive/MyDrive/Coffee_Bean_Detection'),Path('/content/drive/.shortcut-targets-by-id/19430gSQnXwxodRUvcPZmXLwpy9Z_h_2c/Coffee_Bean_Detection')]
if ARCHIVE_ANCHOR is not None: options += [p for p in ARCHIVE_ANCHOR.parents if p.name=='Coffee_Bean_Detection']
options=list(dict.fromkeys(p.resolve() for p in options if p.is_dir()))
matching=[p for p in options if (p/result_rel).is_file() and (p/checkpoint_rel).is_file()]
if not matching: raise FileNotFoundError(f'Folder proyek dengan D0 result/checkpoint tidak ditemukan; kandidat={options}')
PROJECT=matching[0]; D0_RESULT=PROJECT/result_rel; D0_CHECKPOINT=PROJECT/checkpoint_rel
print('PROJECT:',PROJECT); print('ARCHIVE:',ARCHIVE); print('D0 RESULT:',D0_RESULT); print('D0 CHECKPOINT:',D0_CHECKPOINT)


In [ ]:
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU T4 sebelum Run All')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))
check=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_dual_view_audit.py'],cwd=REPO,text=True,capture_output=True)
print(check.stdout)
if check.returncode: print(check.stderr); raise RuntimeError('Dual-view audit tests gagal')


In [ ]:
DATA=WORK/'defectoscafeverde-development-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed={'data.yaml','grouped_audit.json','grouped_manifest.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first=='val' or relative in allowed): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'test').exists() or (DATA/'train').exists(): raise RuntimeError('Hanya validation yang boleh diekstrak')
import yaml
data_yaml=yaml.safe_load((DATA/'data.yaml').read_text()); data_yaml.pop('test',None); data_yaml.pop('train',None)
data_yaml['path']=str(DATA); data_yaml['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(data_yaml,sort_keys=False),encoding='utf-8')
AUDIT=DATA/'grouped_audit.json'; MANIFEST=DATA/'grouped_manifest.json'
for required in (AUDIT,MANIFEST,DATA/'val/images',DATA/'val/labels'):
    if not required.exists(): raise FileNotFoundError(required)
print('VAL:',len(list((DATA/'val/images').glob('*'))),'TRAIN:',(DATA/'train').exists(),'TEST:',(DATA/'test').exists())


In [ ]:
OUT=PROJECT/'experiments/defectoscafeverde-dual-view-audit-v1'; OUT.mkdir(parents=True,exist_ok=True)
SUMMARY=OUT/'dual_view_audit.json'; RECORDS=OUT/'dual_view_records.json'; LOG=OUT/'dual_view_audit_run.log'
command=[sys.executable,'-u','-m','coffee_detector.analysis.defectoscafeverde_dual_view_audit','--data-root',str(DATA),'--grouped-manifest',str(MANIFEST),'--grouped-audit',str(AUDIT),'--d0-result',str(D0_RESULT),'--d0-checkpoint',str(D0_CHECKPOINT),'--output',str(SUMMARY),'--records-output',str(RECORDS),'--device','0']
print('MENJALANKAN VALIDATION-ONLY DUAL-VIEW AUDIT | log=',LOG,flush=True)
with LOG.open('w',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
shown=None
while process.poll() is None:
    lines=LOG.read_text(errors='replace').splitlines() if LOG.is_file() else []
    status=[line for line in lines if line.startswith('DUAL VIEW AUDIT')]
    if status and status[-1]!=shown: print(status[-1],flush=True); shown=status[-1]
    time.sleep(30)
print('\n'.join(LOG.read_text(errors='replace').splitlines()[-100:]))
if process.returncode: raise RuntimeError(f'Audit gagal: {process.returncode}; log={LOG}')
if not SUMMARY.is_file(): raise FileNotFoundError(SUMMARY)


In [ ]:
from IPython.display import display
import pandas as pd
result=json.loads(SUMMARY.read_text()); summary=result['summary']
overall=[{'metric':key,'value':summary[key]} for key in ('side_a_accuracy','side_b_accuracy','best_single_side_accuracy','paper_max_confidence_accuracy','pair_oracle_accuracy','paper_gain_over_best_single_side','pair_oracle_gain_over_best_single_side','learned_fusion_headroom_over_paper_rule')]
display(pd.DataFrame(overall).style.format({'value':'{:+.2%}'}))
class_rows=[{'class_name':name,**values} for name,values in summary['classwise'].items()]
display(pd.DataFrame(class_rows).sort_values('exactly_one_side_correct',ascending=False).style.format({key:'{:.2%}' for key in ('side_a_accuracy','side_b_accuracy','paper_max_confidence_accuracy','pair_oracle_accuracy')}))
print('MANIFEST:',result['manifest'],'| ELIGIBLE FRACTION:',result['eligible_group_fraction'])
print('REJECTED:',result['rejected_groups'])
print('PAIR COUNTS:',{key:summary[key] for key in ('both_sides_correct','exactly_one_side_correct','neither_side_correct','complementary_classes')})
print('WORST CLASS:',result['reference_worst_class'],'| COMPLEMENTARY GROUPS:',result['reference_worst_class_complementary_groups'])
print('CRITERIA:',result['criteria'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_images_accessed'])
print('SUMMARY:',SUMMARY,'| RECORDS:',RECORDS)
print('Kirim dua tabel, MANIFEST, REJECTED, PAIR COUNTS, WORST CLASS, CRITERIA, DECISION, dan NEXT. Jangan training atau membuka test.')
